## 准备数据

In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [10]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [11]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal(shape=[28*28, 100]))
        self.W2 = tf.Variable(tf.random.normal(shape=[100, 10]))
        self.b1 = tf.Variable(tf.random.normal(shape=[1,100]))
        self.b2 = tf.Variable(tf.random.normal(shape=[1, 10]))
        
    @tf.function
    def __call__(self, x):
        with tf.GradientTape() as tape:
            ####################
            '''实现模型函数体，返回未归一化的logits'''
            ####################
            x = tf.reshape(x, [-1, 28*28])
            # bias1 = np.ones(shape=[x.shape[0], 1])
            # x = tf.concat([x, bias1], axis=1)
            # self.tmpw1 = tf.concat([self.W1, self.b1], axis=0)
            
            self.h1 = tf.matmul(x, self.W1) + self.b1
            self.h1_relu = tf.nn.relu(self.h1)
            
            # bias2 = tf.ones(shape=[self.h1_relu.shape[0], 1])
            # self.h1_relu = tf.concat([self.h1_relu, bias2], axis=1)
            # self.tmpw2 = tf.concat([self.W2, self.b2], axis=0)
            self.h2 = tf.matmul(self.h1_relu, self.W2) + self.b2
            self.h2_relu = tf.nn.relu(self.h2)
            self.h2_soft = tf.nn.softmax(self.h2_relu)
            self.h2_log = tf.math.log(self.h2_soft + 1e-12)
            # logits = self.h2_log        
        return self.h2_log
        

model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [12]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    # trainable_vars = [model.W1, model.W2]
    grads = tape.gradient(loss, trainable_vars)
    # for g, v in zip(grads, trainable_vars):
    #     v.assign_sub(0.2*g)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [13]:
train_data, test_data = mnist_dataset()
for epoch in range(500):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 23.57401 ; accuracy 0.08473333
epoch 1 : loss 23.380388 ; accuracy 0.08891667
epoch 2 : loss 23.182129 ; accuracy 0.0929
epoch 3 : loss 22.979704 ; accuracy 0.097
epoch 4 : loss 22.773859 ; accuracy 0.100766666
epoch 5 : loss 22.565014 ; accuracy 0.104783334
epoch 6 : loss 22.353767 ; accuracy 0.108783334
epoch 7 : loss 22.140715 ; accuracy 0.11255
epoch 8 : loss 21.924637 ; accuracy 0.11728334
epoch 9 : loss 21.707333 ; accuracy 0.120916665
epoch 10 : loss 21.48787 ; accuracy 0.12508333
epoch 11 : loss 21.264223 ; accuracy 0.12928334
epoch 12 : loss 21.035372 ; accuracy 0.13301666
epoch 13 : loss 20.801376 ; accuracy 0.1363
epoch 14 : loss 20.561453 ; accuracy 0.13995
epoch 15 : loss 20.314451 ; accuracy 0.14333333
epoch 16 : loss 20.056278 ; accuracy 0.1462
epoch 17 : loss 19.787754 ; accuracy 0.14936666
epoch 18 : loss 19.50917 ; accuracy 0.15226667
epoch 19 : loss 19.218029 ; accuracy 0.15531667
epoch 20 : loss 18.913656 ; accuracy 0.15813333
epoch 21 : loss 18.59383